In [ ]:
#

import re
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

# ================= CONFIG =================
LABEL_CSV = "../LiverFibrosisCohort/LiverFibrosisCohort_MasterTable_Proccessed_527.csv"

ID_COLUMN = "Image ID"
LABEL_COLUMN = "Fibrosis_Label"
SITE_COLUMN = "SITE"

DEV_SITES = ["CCHMC", "WISC", "MICH"]
EXTERNAL_SITE = "NYU"

SEED = 42
N_SPLITS = 5
EPOCHS = 100
LR = 5e-4
WEIGHT_DECAY = 0
FIXED_THRESHOLD = 0.54
FINAL_RETRAIN_VAL_FRAC = 0.20
CV_INNER_VAL_FRAC = 0.15
EARLY_STOPPING_PATIENCE = 15
MIN_EPOCHS = 10

BATCH_SIZE = 32
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = DEVICE.startswith("cuda")

# ================= FEATURE SETS: CLINICAL ONLY =================
COMMON_NUMERIC_COLS = ["AGE_years", "BMI"]
COMMON_CATEGORICAL_COLS = ["SEX", "RACE", "ETHNICITY"]

CLINICAL_NUMERIC_COLS = [
    "TBIL (mg/dL)",
    "ALT (U/L)",
    "A1C (%)",
    "GLUC (mg/dL)",
    "DBIL (mg/dL)",
    "ALB (g/dL)",
    "PLT (K/uL)",
    "AST (U/L)",
    "ALK (U/L)",
    "HCT (%)",
] + COMMON_NUMERIC_COLS

CLINICAL_HISTORY_COLS = [
    "Alcohol_Liver_Disease",
    "Fatty_Liver_Disease(NAFLD_NASH)",
    "Primary_Sclerosing_Cholangitis(PSC)",
    "Liver_Transplantation",
    "Primary_Biliary_Cirrhosis(PBC)",
    "Hepatitis_B_C",
    "Autoimmune_Hepatitis(AIH)",
    "Diabetes_Type2",
]

CLINICAL_CATEGORICAL_COLS = COMMON_CATEGORICAL_COLS + CLINICAL_HISTORY_COLS

MODEL_NAME = "Clinical"
NUMERIC_COLS = CLINICAL_NUMERIC_COLS
CATEGORICAL_COLS = CLINICAL_CATEGORICAL_COLS

# ================= REPRODUCIBILITY =================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ================= HELPERS =================
def normalize_id(pid):
    pid = str(pid)

    m = re.match(r"PT-(\d{3}-\d{4})-", pid)
    if m:
        return m.group(1)

    m = re.match(r"PT-(M\d+)-", pid)
    if m:
        return m.group(1)

    return pid


def get_site_from_id(pid):
    """
    Fallback only if SITE column is unavailable.
    Avoid using this when SITE exists.
    """
    pid = str(pid)

    if re.match(r"^\d{3}-\d{4}$", pid) or pid.startswith("M001"):
        return "CCHMC"

    if pid.startswith("GJ"):
        return "WISC"

    if pid.startswith("C_"):
        return "MICH"

    if re.match(r"^\d+$", pid):
        return "NYU"

    return "UNKNOWN"


def clean_site(x):
    s = str(x).strip().upper()

    if s in ["CCHMC1", "CCHMC2"] or s.startswith("CCHMC"):
        return "CCHMC"
    if s in ["UW1", "UW2"] or s.startswith("UW"):
        return "WISC"
    if s == "NYU" or s.startswith("NYU"):
        return "NYU"
    if s == "UM" or s.startswith("UM"):
        return "MICH"

    return "UNKNOWN"


def binarize_fibrosis(x):
    """
    Supports:
    - already-binary labels: 0/1
    - fibrosis stages: 0/1 -> 0, 2/3/4 -> 1
    """
    if pd.isna(x):
        return np.nan

    val = float(x)
    if val in [0, 1]:
        return int(val)
    if val in [2, 3, 4]:
        return 1
    return np.nan


def mean_std(series):
    vals = pd.Series(series).dropna().values
    if len(vals) == 0:
        return np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), 0.0
    return float(np.mean(vals)), float(np.std(vals, ddof=1))

# ================= DATA LOADING =================
def load_master_table():
    df = pd.read_csv(LABEL_CSV)
    df["id"] = df[ID_COLUMN].astype(str)

    if "AGE_years" not in df.columns and "AGE_days" in df.columns:
        df["AGE_years"] = pd.to_numeric(df["AGE_days"], errors="coerce") / 365.25

    df["label"] = df[LABEL_COLUMN].apply(binarize_fibrosis)
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)

    if SITE_COLUMN in df.columns:
        df["site"] = df[SITE_COLUMN].apply(clean_site)
    else:
        df["site"] = df["id"].apply(get_site_from_id)

    df = df[df["site"].isin(DEV_SITES + [EXTERNAL_SITE])].copy()
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    return df


def validate_columns(df, cols, model_name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{model_name}: missing columns: {missing}")

# ================= PREPROCESSING =================
def fit_median_imputer(X_train):
    medians = np.nanmedian(X_train, axis=0)
    medians = np.where(np.isnan(medians), 0.0, medians)
    return medians


def apply_median_imputer(X, medians):
    X = X.copy()
    inds = np.where(np.isnan(X))
    X[inds] = medians[inds[1]]
    return X


def fit_numeric_preprocessor(df, numeric_cols):
    if len(numeric_cols) == 0:
        return np.empty((len(df), 0), dtype=np.float32), {
            "medians": np.array([], dtype=np.float32),
            "scaler": None,
            "numeric_cols": numeric_cols,
        }

    X = df[numeric_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    X[np.isinf(X)] = np.nan

    medians = fit_median_imputer(X)
    X = apply_median_imputer(X, medians)

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X.astype(np.float32), {
        "medians": medians,
        "scaler": scaler,
        "numeric_cols": numeric_cols,
    }


def transform_numeric(df, preprocessor):
    if len(preprocessor["numeric_cols"]) == 0:
        return np.empty((len(df), 0), dtype=np.float32)

    X = df[preprocessor["numeric_cols"]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    X[np.isinf(X)] = np.nan
    X = apply_median_imputer(X, preprocessor["medians"])
    X = preprocessor["scaler"].transform(X)
    return X.astype(np.float32)


def fit_categorical_encoder(df, categorical_cols):
    categories = {}
    encoded_frames = []
    encoded_cols = []

    for col in categorical_cols:
        ser = df[col].fillna("Missing").astype(str)
        cats = sorted(ser.unique().tolist())
        categories[col] = cats

        dummies = pd.get_dummies(pd.Categorical(ser, categories=cats), prefix=col)
        encoded_frames.append(dummies)
        encoded_cols.extend(dummies.columns.tolist())

    if encoded_frames:
        X_cat = pd.concat(encoded_frames, axis=1)
    else:
        X_cat = pd.DataFrame(index=df.index)

    return X_cat.to_numpy(dtype=np.float32), {
        "categorical_cols": categorical_cols,
        "categories": categories,
        "encoded_cols": encoded_cols,
    }


def transform_categorical(df, encoder):
    frames = []

    for col in encoder["categorical_cols"]:
        ser = df[col].fillna("Missing").astype(str)
        cats = encoder["categories"][col]
        dummies = pd.get_dummies(pd.Categorical(ser, categories=cats), prefix=col)
        frames.append(dummies)

    if frames:
        X_cat = pd.concat(frames, axis=1)
        X_cat = X_cat.reindex(columns=encoder["encoded_cols"], fill_value=0.0)
    else:
        X_cat = pd.DataFrame(index=df.index)

    return X_cat.to_numpy(dtype=np.float32)


def fit_feature_preprocessor(train_df, numeric_cols, categorical_cols):
    X_num, num_pre = fit_numeric_preprocessor(train_df, numeric_cols)
    X_cat, cat_pre = fit_categorical_encoder(train_df, categorical_cols)

    X = np.hstack([X_num, X_cat]).astype(np.float32)

    preprocessor = {
        "numeric": num_pre,
        "categorical": cat_pre,
        "feature_names": numeric_cols + cat_pre["encoded_cols"],
    }
    return X, preprocessor


def transform_features(df, preprocessor):
    X_num = transform_numeric(df, preprocessor["numeric"])
    X_cat = transform_categorical(df, preprocessor["categorical"])
    return np.hstack([X_num, X_cat]).astype(np.float32)

# ================= METRICS =================
def safe_roc_auc(y_true, y_prob):
    try:
        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return np.nan


def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    auc = safe_roc_auc(y_true, y_prob)

    return {
        "balanced_accuracy": balanced_acc,
        "auc": auc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def make_sample_weights(y):
    y = np.asarray(y).astype(int)
    n = len(y)
    n0 = max((y == 0).sum(), 1)
    n1 = max((y == 1).sum(), 1)
    w0 = n / (2.0 * n0)
    w1 = n / (2.0 * n1)
    return np.where(y == 0, w0, w1).astype(np.float32)

# ================= MODEL =================
class SimpleLinearModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

# ================= DATALOADER UTILS =================
def build_train_loader(X, y, batch_size, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y.astype(np.float32), dtype=torch.float32)
    w_t = torch.tensor(make_sample_weights(y), dtype=torch.float32)

    ds = TensorDataset(X_t, y_t, w_t)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )


def build_eval_loader(X, y=None, batch_size=256):
    X_t = torch.tensor(X, dtype=torch.float32)

    if y is None:
        ds = TensorDataset(X_t)
    else:
        y_t = torch.tensor(y.astype(np.float32), dtype=torch.float32)
        ds = TensorDataset(X_t, y_t)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )


def predict_probs(model, X, batch_size=256):
    loader = build_eval_loader(X, y=None, batch_size=batch_size)
    model.eval()

    probs_all = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
            logits = model(xb)
            probs = torch.sigmoid(logits)
            probs_all.append(probs.cpu().numpy())

    if len(probs_all) == 0:
        return np.array([], dtype=np.float32)

    return np.concatenate(probs_all).astype(np.float32)


def compute_val_loss(model, X_val, y_val, batch_size=256):
    loader = build_eval_loader(X_val, y=y_val, batch_size=batch_size)
    loss_fn = nn.BCEWithLogitsLoss(reduction="sum")

    model.eval()
    total_loss = 0.0
    total_n = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
            yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)

            logits = model(xb)
            loss = loss_fn(logits, yb)

            total_loss += float(loss.item())
            total_n += int(yb.numel())

    if total_n == 0:
        return np.inf

    return total_loss / total_n


def train_model(
    X_train,
    y_train,
    input_dim,
    X_val=None,
    y_val=None,
    early_stopping_patience=None,
    min_epochs=1,
    batch_size=32,
):
    train_loader = build_train_loader(
        X_train,
        y_train,
        batch_size=batch_size,
        shuffle=True,
    )

    model = SimpleLinearModel(input_dim=input_dim).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    loss_fn = nn.BCEWithLogitsLoss(reduction="none")

    use_early_stopping = (
        X_val is not None
        and y_val is not None
        and early_stopping_patience is not None
    )

    best_state_dict = None
    best_epoch = 0
    best_val_loss = np.inf
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()

        for xb, yb, wb in train_loader:
            xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
            yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)
            wb = wb.to(DEVICE, non_blocking=PIN_MEMORY)

            logits = model(xb)
            loss_vec = loss_fn(logits, yb)
            loss = (loss_vec * wb).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if use_early_stopping:
            val_loss_value = compute_val_loss(model, X_val, y_val, batch_size=max(256, batch_size))

            if best_epoch == 0 or val_loss_value < best_val_loss:
                best_val_loss = val_loss_value
                best_epoch = epoch
                best_state_dict = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epoch >= min_epochs and epochs_without_improvement >= early_stopping_patience:
                break

    if use_early_stopping and best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model

# ================= TRAIN / EVAL =================
def print_cohort_info(df, name):
    print(f"{name}: n={len(df)}")
    if len(df) > 0:
        print(df["label"].value_counts().sort_index())
    print()


def make_inner_train_val_split(train_df, val_frac=0.15):
    try:
        tr_df, val_df = train_test_split(
            train_df,
            test_size=val_frac,
            random_state=SEED,
            stratify=train_df["label"].values.astype(int),
        )
    except ValueError:
        tr_df, val_df = train_test_split(
            train_df,
            test_size=val_frac,
            random_state=SEED,
            shuffle=True,
        )

    return tr_df.reset_index(drop=True), val_df.reset_index(drop=True)


def run_dev_cv(df, numeric_cols, categorical_cols, model_name):
    y = df["label"].values.astype(int)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    fold_rows = []

    for fold_idx, (tr, val) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
        outer_train_df = df.iloc[tr].reset_index(drop=True)
        outer_val_df = df.iloc[val].reset_index(drop=True)

        inner_train_df, inner_stop_df = make_inner_train_val_split(
            outer_train_df,
            val_frac=CV_INNER_VAL_FRAC
        )

        X_train, preprocessor = fit_feature_preprocessor(
            inner_train_df,
            numeric_cols,
            categorical_cols
        )
        X_stop = transform_features(inner_stop_df, preprocessor)
        X_val = transform_features(outer_val_df, preprocessor)

        y_train = inner_train_df["label"].values.astype(np.float32)
        y_stop = inner_stop_df["label"].values.astype(int)
        y_val = outer_val_df["label"].values.astype(int)

        model = train_model(
            X_train=X_train,
            y_train=y_train,
            input_dim=X_train.shape[1],
            X_val=X_stop,
            y_val=y_stop,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            min_epochs=MIN_EPOCHS,
            batch_size=BATCH_SIZE,
        )

        stop_probs = predict_probs(model, X_stop, batch_size=max(256, BATCH_SIZE))
        stop_metrics = compute_binary_metrics(y_stop, stop_probs, threshold=FIXED_THRESHOLD)

        val_probs = predict_probs(model, X_val, batch_size=max(256, BATCH_SIZE))
        metrics = compute_binary_metrics(y_val, val_probs, threshold=FIXED_THRESHOLD)

        row = {
            "fold": fold_idx,
            "n_train": len(inner_train_df),
            "n_stop": len(inner_stop_df),
            "n_val": len(outer_val_df),
            "threshold": FIXED_THRESHOLD,
            "stop_ba": stop_metrics["balanced_accuracy"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "auc": metrics["auc"],
            "sensitivity": metrics["sensitivity"],
            "specificity": metrics["specificity"],
            "tp": metrics["tp"],
            "tn": metrics["tn"],
            "fp": metrics["fp"],
            "fn": metrics["fn"],
        }
        fold_rows.append(row)

        print(
            f"{model_name} | DEV CV fold {fold_idx}: "
            f"thr={FIXED_THRESHOLD:.3f}, "
            f"BA={metrics['balanced_accuracy']:.4f}, "
            f"AUC={metrics['auc']:.4f}, "
            f"Sen={metrics['sensitivity']:.4f}, "
            f"Spe={metrics['specificity']:.4f}"
        )

    fold_df = pd.DataFrame(fold_rows)

    print(f"\n{model_name} | DEV CV summary")
    print(f"Thr: {mean_std(fold_df['threshold'])[0]:.4f} ± {mean_std(fold_df['threshold'])[1]:.4f}")
    print(f"BA : {mean_std(fold_df['balanced_accuracy'])[0]:.4f} ± {mean_std(fold_df['balanced_accuracy'])[1]:.4f}")
    print(f"Sen: {mean_std(fold_df['sensitivity'])[0]:.4f} ± {mean_std(fold_df['sensitivity'])[1]:.4f}")
    print(f"Spe: {mean_std(fold_df['specificity'])[0]:.4f} ± {mean_std(fold_df['specificity'])[1]:.4f}")
    print(f"AUC: {mean_std(fold_df['auc'])[0]:.4f} ± {mean_std(fold_df['auc'])[1]:.4f}")
    print()

    return fold_df


def make_final_retrain_split(dev_df, val_frac=0.20):
    try:
        train_df, val_df = train_test_split(
            dev_df,
            test_size=val_frac,
            random_state=SEED,
            stratify=dev_df["label"].values.astype(int),
        )
    except ValueError:
        train_df, val_df = train_test_split(
            dev_df,
            test_size=val_frac,
            random_state=SEED,
            shuffle=True,
        )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def retrain_on_dev_and_test_external(dev_df, external_df, numeric_cols, categorical_cols, model_name):
    retrain_train_df, retrain_val_df = make_final_retrain_split(dev_df, FINAL_RETRAIN_VAL_FRAC)

    X_train, preprocessor = fit_feature_preprocessor(
        retrain_train_df,
        numeric_cols,
        categorical_cols
    )
    X_val = transform_features(retrain_val_df, preprocessor)

    y_train = retrain_train_df["label"].values.astype(np.float32)
    y_val = retrain_val_df["label"].values.astype(int)

    model = train_model(
        X_train=X_train,
        y_train=y_train,
        input_dim=X_train.shape[1],
        X_val=X_val,
        y_val=y_val,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        min_epochs=MIN_EPOCHS,
        batch_size=BATCH_SIZE,
    )

    val_probs = predict_probs(model, X_val, batch_size=max(256, BATCH_SIZE))
    val_metrics = compute_binary_metrics(y_val, val_probs, threshold=FIXED_THRESHOLD)

    X_ext = transform_features(external_df, preprocessor)
    y_ext = external_df["label"].values.astype(int)

    ext_probs = predict_probs(model, X_ext, batch_size=max(256, BATCH_SIZE))
    ext_metrics = compute_binary_metrics(y_ext, ext_probs, threshold=FIXED_THRESHOLD)

    print(f"{model_name} | External {EXTERNAL_SITE}")
    print(f"n={len(external_df)}")
    print(f"Fixed threshold: {FIXED_THRESHOLD:.4f} (retrain-val BA={val_metrics['balanced_accuracy']:.4f})")
    print(f"BA : {ext_metrics['balanced_accuracy']:.4f}")
    print(f"Sen: {ext_metrics['sensitivity']:.4f}")
    print(f"Spe: {ext_metrics['specificity']:.4f}")
    print(f"AUC: {ext_metrics['auc']:.4f}")
    print(
        f"TP={ext_metrics['tp']}, TN={ext_metrics['tn']}, "
        f"FP={ext_metrics['fp']}, FN={ext_metrics['fn']}"
    )
    print()

    return ext_metrics

# ================= MAIN =================
def main():
    set_seed(SEED)
    print(f"Using device: {DEVICE}\n")

    df = load_master_table()
    print("Cleaned site counts:")
    print(df["site"].value_counts(dropna=False))
    print()

    print("DEV counts:")
    print(df[df["site"].isin(DEV_SITES)]["site"].value_counts(dropna=False))
    print()

    print(f"{EXTERNAL_SITE} count:")
    print((df["site"] == EXTERNAL_SITE).sum())
    print()

    dev_df = df[df["site"].isin(DEV_SITES)].copy().reset_index(drop=True)
    external_df = df[df["site"] == EXTERNAL_SITE].copy().reset_index(drop=True)

    print_cohort_info(dev_df, f"DEV cohort ({' + '.join(DEV_SITES)})")
    print_cohort_info(external_df, f"External cohort ({EXTERNAL_SITE})")

    validate_columns(df, NUMERIC_COLS + CATEGORICAL_COLS, MODEL_NAME)

    print("=" * 80)
    print(f"Model: {MODEL_NAME}")
    print("Numeric columns:", NUMERIC_COLS)
    print("Categorical columns:", CATEGORICAL_COLS)
    print(f"Batch size: {BATCH_SIZE}")
    print()

    _ = run_dev_cv(
        df=dev_df,
        numeric_cols=NUMERIC_COLS,
        categorical_cols=CATEGORICAL_COLS,
        model_name=MODEL_NAME,
    )

    _ = retrain_on_dev_and_test_external(
        dev_df=dev_df,
        external_df=external_df,
        numeric_cols=NUMERIC_COLS,
        categorical_cols=CATEGORICAL_COLS,
        model_name=MODEL_NAME,
    )

    print("=" * 80)
    print("DONE")


if __name__ == "__main__":
    main()

Using device: cuda

Cleaned site counts:
site
CCHMC    236
WISC     165
NYU       80
MICH      46
Name: count, dtype: int64

DEV counts:
site
CCHMC    236
WISC     165
MICH      46
Name: count, dtype: int64

NYU count:
80

DEV cohort (CCHMC + WISC + MICH): n=447
label
0    137
1    310
Name: count, dtype: int64

External cohort (NYU): n=80
label
0    15
1    65
Name: count, dtype: int64

Model: Clinical
Numeric columns: ['TBIL (mg/dL)', 'ALT (U/L)', 'A1C (%)', 'GLUC (mg/dL)', 'DBIL (mg/dL)', 'ALB (g/dL)', 'PLT (K/uL)', 'AST (U/L)', 'ALK (U/L)', 'HCT (%)', 'AGE_years', 'BMI']
Categorical columns: ['SEX', 'RACE', 'ETHNICITY', 'Alcohol_Liver_Disease', 'Fatty_Liver_Disease(NAFLD_NASH)', 'Primary_Sclerosing_Cholangitis(PSC)', 'Liver_Transplantation', 'Primary_Biliary_Cirrhosis(PBC)', 'Hepatitis_B_C', 'Autoimmune_Hepatitis(AIH)', 'Diabetes_Type2']
Batch size: 32

Clinical | DEV CV fold 1: thr=0.540, BA=0.6440, AUC=0.6538, Sen=0.6452, Spe=0.6429
Clinical | DEV CV fold 2: thr=0.540, BA=0.6169,